# Test du RaspBot V2 - LOG635 Labo 1

**Mode d'emploi**
1. Exécute les cellules de la section **1. Préparation** une seule fois (`Shift+Entrée`).
2. Dans la section **2. Tests**, exécute seulement les tests que tu veux, dans l'ordre que tu veux.
3. En cas de problème, exécute la cellule **Arrêt d'urgence** tout en bas.

⚠️ Un seul notebook actif à la fois : arrête le noyau des autres notebooks (*Kernel → Shut Down Kernel*).

**Comment le robot est contrôlé**

Le Raspberry Pi ne pilote pas directement les moteurs : il envoie des ordres à une **carte d'extension**
(adresse I2C `0x2B`) par le **bus I2C**. La bibliothèque Yahboom `Raspbot_Lib` cache ces détails :
chaque fonction `bot.Ctrl_...()` écrit quelques octets dans un « registre » de la carte
(ex. registre `0x01` = moteurs, `0x02` = servos, `0x03` = LED, `0x06` = buzzer),
et `bot.read_data_array(registre, n)` lit `n` octets (capteurs).

## 1. Préparation

### 1.1 Imports et création du robot
Cette cellule charge les bibliothèques et crée l'objet `bot` qui sert à parler au robot.

In [1]:
# ----- Bibliothèques standard de Python -----
import os                      # gestion des fichiers et dossiers (créer le dossier photos/)
import sys                     # permet de modifier le chemin de recherche des modules
import time                    # time.sleep() pour attendre, time.time() pour mesurer le temps
from datetime import datetime  # date et heure, pour nommer les photos

# ----- Bibliothèques externes -----
import cv2                                  # OpenCV : lecture de la caméra et enregistrement des images
from IPython.display import Image, display  # affichage d'une image directement sous la cellule Jupyter

# Le module McLumk_Wheel_Sports n'est pas installé comme un paquet Python :
# il se trouve dans /home/pi/project_demo/lib. On ajoute ce dossier à la liste
# des endroits où Python cherche les modules, sinon l'import échouerait.
sys.path.append('/home/pi/project_demo/lib')

# McLumk_Wheel_Sports (fourni par Yahboom) contient :
#   - bot : l'objet Raspbot() déjà créé, qui communique avec la carte moteurs par I2C
#   - des fonctions de déplacement pour les roues mecanum (roues à rouleaux en biais)
#     qui calculent la vitesse de chacune des 4 roues selon la direction voulue.
# On réutilise SON objet bot plutôt que d'en créer un deuxième, pour éviter
# d'avoir deux objets qui envoient des ordres au robot en même temps.
from McLumk_Wheel_Sports import (bot,
                                 move_forward,   # avancer
                                 move_backward,  # reculer
                                 move_left,      # glisser à gauche (sans tourner)
                                 move_right,     # glisser à droite (sans tourner)
                                 rotate_left,    # pivoter sur place vers la gauche
                                 rotate_right,   # pivoter sur place vers la droite
                                 stop_robot)     # mettre les 4 moteurs à vitesse 0

# Numéros des couleurs de la barre de LED, tels que définis par la carte Yahboom.
# range(7) donne 0, 1, 2, 3, 4, 5, 6 : ROUGE = 0, VERT = 1, ..., BLANC = 6.
ROUGE, VERT, BLEU, JAUNE, VIOLET, CYAN, BLANC = range(7)

# Vitesse utilisée pour les déplacements au sol (0 à 255).
# Trop basse (ex. 50), les roues n'arrivent pas à faire bouger le robot
# (frottement, poids, batterie faible). Les démos Yahboom utilisent 100.
VITESSE_SOL = 100


def tout_arreter():
    """Remet le robot dans un état sûr : moteurs, buzzer et lumières éteints."""
    stop_robot()               # vitesse 0 sur les 4 moteurs
    bot.Ctrl_BEEP_Switch(0)    # 0 = buzzer éteint
    bot.Ctrl_WQ2812_ALL(0, 0)  # 1er paramètre 0 = LED éteintes (la couleur est ignorée)

print("Robot prêt")

Robot prêt


### 1.2 Buzzer, lumières, servos, moteurs
**Fonctions utilisées :**

| Fonction | Paramètres |
|---|---|
| `bot.Ctrl_BEEP_Switch(etat)` | 1 = bip, 0 = silence |
| `bot.Ctrl_WQ2812_ALL(etat, couleur)` | etat 1/0, couleur 0 à 6 |
| `bot.Ctrl_Servo(id, angle)` | id 1 = gauche/droite (0-180°), id 2 = haut/bas (0-110°) |
| `bot.Ctrl_Muto(moteur, vitesse)` | moteur 0 à 3, vitesse -255 (arrière) à 255 (avant) |

In [2]:
def test_buzzer():
    """Fait un bip court de 0,3 seconde."""
    bot.Ctrl_BEEP_Switch(1)   # allume le buzzer
    time.sleep(0.3)           # le laisse sonner 0,3 s (le programme attend ici)
    bot.Ctrl_BEEP_Switch(0)   # l'éteint


def test_lumieres():
    """Allume la barre de LED dans chacune des 7 couleurs, 0,5 s par couleur."""
    for couleur in (ROUGE, VERT, BLEU, JAUNE, VIOLET, CYAN, BLANC):
        bot.Ctrl_WQ2812_ALL(1, couleur)  # 1 = allumer toutes les LED dans cette couleur
        time.sleep(0.5)
    bot.Ctrl_WQ2812_ALL(0, 0)            # tout éteindre à la fin


def test_servos():
    """Bouge le support de caméra : gauche/droite puis haut/bas, et revient au centre."""
    # Servo 1 = rotation horizontale (0 à 180°, 90 = centre)
    for angle in (45, 135, 90):
        bot.Ctrl_Servo(1, angle)
        time.sleep(0.6)  # un servo met un peu de temps à atteindre sa position
    # Servo 2 = inclinaison verticale (limité à 110° par la bibliothèque pour ne pas forcer)
    for angle in (30, 100, 90):
        bot.Ctrl_Servo(2, angle)
        time.sleep(0.6)


def test_moteurs(vitesse=60):
    """Fait tourner chaque roue seule pendant 1 s pour identifier quel numéro correspond à quelle roue.
    À faire avec le robot SOULEVÉ (roues dans le vide)."""
    # try / finally : le bloc finally s'exécute TOUJOURS, même si une erreur survient
    # ou si on interrompt la cellule avec ■. Ça garantit que les moteurs s'arrêtent.
    try:
        # enumerate() donne à la fois le numéro (0, 1, 2, 3) et le nom de chaque roue
        noms = ("avant gauche", "arrière gauche", "avant droit", "arrière droit")
        for moteur, nom in enumerate(noms):
            print(f"Moteur {moteur} ({nom})")  # f"..." insère les variables dans le texte
            bot.Ctrl_Muto(moteur, vitesse)     # démarre cette roue seulement
            time.sleep(1)
            bot.Ctrl_Muto(moteur, 0)           # l'arrête
            time.sleep(0.3)                    # petite pause avant la roue suivante
    finally:
        stop_robot()


def test_deplacements(vitesse=VITESSE_SOL, duree=1):
    """Essaie les 6 déplacements de base, chacun pendant `duree` secondes.
    À faire avec le robot AU SOL et de l'espace libre autour."""
    # Liste de paires (nom affiché, fonction à appeler).
    # En Python, une fonction est une valeur comme une autre : on peut la mettre
    # dans une liste et l'appeler plus tard avec mouvement(vitesse).
    mouvements = (("avancer", move_forward),
                  ("reculer", move_backward),
                  ("glisser à gauche", move_left),
                  ("glisser à droite", move_right),
                  ("tourner à gauche", rotate_left),
                  ("tourner à droite", rotate_right))
    try:
        for nom, mouvement in mouvements:
            print(nom)
            mouvement(vitesse)  # lance le mouvement (le robot continue tant qu'on ne l'arrête pas)
            time.sleep(duree)   # laisse le robot bouger pendant `duree` secondes
            stop_robot()
            time.sleep(0.5)     # pause pour bien voir la différence entre deux mouvements
    finally:
        stop_robot()

### 1.3 Capteurs : ultrason et suivi de ligne
Les capteurs se **lisent** avec `bot.read_data_array(registre, nombre_d_octets)`, qui retourne une liste d'octets (0 à 255).
- **Ultrason** : la distance en mm tient sur 2 octets (registre `0x1b` = octet haut, `0x1a` = octet bas).
- **Suivi de ligne** : les 4 capteurs infrarouges sont regroupés dans 1 seul octet (registre `0x0a`), 1 bit par capteur.

In [3]:
def lire_distance_mm():
    """Retourne la distance mesurée par le capteur ultrason, en millimètres.
    Le capteur doit avoir été allumé avant avec bot.Ctrl_Ulatist_Switch(1)."""
    haut = bot.read_data_array(0x1b, 1)[0]  # [0] : on prend le seul octet de la liste
    bas = bot.read_data_array(0x1a, 1)[0]
    # Un octet ne va que jusqu'à 255. Pour des distances plus grandes, la valeur
    # est coupée en deux octets. On les recolle :
    #   haut << 8  décale l'octet haut de 8 bits (= multiplier par 256)
    #   | bas      ajoute l'octet bas dans les 8 bits libérés
    # Exemple : haut = 1, bas = 44  ->  1 * 256 + 44 = 300 mm
    return haut << 8 | bas


def test_ultrason(n=10):
    """Affiche n mesures de distance, une toutes les 0,3 s."""
    bot.Ctrl_Ulatist_Switch(1)  # allume le capteur ultrason
    time.sleep(0.5)             # lui laisse le temps de faire une première mesure
    try:
        for _ in range(n):      # _ : on n'a pas besoin du numéro de la boucle
            print(f"Distance : {lire_distance_mm()} mm")
            time.sleep(0.3)
    finally:
        bot.Ctrl_Ulatist_Switch(0)  # éteint le capteur (économise la batterie)


def lire_capteurs_ligne():
    """Retourne l'état des 4 capteurs de ligne : [extrême gauche, gauche, droite, extrême droite].
    Chaque valeur vaut 0 ou 1 (selon que le capteur voit une surface sombre ou claire)."""
    track = bot.read_data_array(0x0a, 1)[0]  # un octet, par exemple 0b0110 = 6
    # Chaque capteur correspond à un bit de l'octet. Pour extraire un bit :
    #   track >> 3  décale l'octet de 3 bits vers la droite (le bit 3 arrive en position 0)
    #   & 1         garde uniquement ce dernier bit (0 ou 1)
    # Exemple avec track = 6 (en binaire 0110) : bits 3, 2, 1, 0 = 0, 1, 1, 0
    return [(track >> 3) & 1,  # bit 3 : extrême gauche
            (track >> 2) & 1,  # bit 2 : gauche
            (track >> 1) & 1,  # bit 1 : droite
            track & 1]         # bit 0 : extrême droite


def test_capteurs_ligne(n=10):
    """Affiche n lectures des capteurs de ligne. Passe-les sur une ligne noire pour voir les valeurs changer."""
    for _ in range(n):
        print(f"Capteurs ligne : {lire_capteurs_ligne()}")
        time.sleep(0.3)

### 1.4 Avancer
Le robot n'a **pas de capteur de distance parcourue** : on le fait avancer pendant un certain **temps**.
Pour avancer d'une distance précise, il faut **calibrer** `CM_PAR_SECONDE` (voir la cellule).

In [4]:
def avancer(vitesse=VITESSE_SOL, duree=1.0):
    """Fait avancer le robot en ligne droite pendant `duree` secondes.

    vitesse : 0 à 255 (trop basse, le robot ne bouge pas au sol : voir VITESSE_SOL)
    duree   : temps en secondes
    """
    # int() convertit en nombre entier (la carte n'accepte pas les décimales),
    # min(255, ...) empêche de dépasser 255, max(0, ...) empêche d'aller sous 0.
    vitesse = max(0, min(255, int(vitesse)))
    try:
        move_forward(vitesse)  # démarre les 4 roues vers l'avant
        time.sleep(duree)      # le robot roule pendant ce temps
    finally:
        # Exécuté dans tous les cas, même si la cellule est interrompue (■) :
        # le robot ne peut pas rester bloqué en marche.
        stop_robot()


def avancer_securise(vitesse=VITESSE_SOL, duree=2.0, distance_min=150):
    """Avance pendant `duree` secondes, mais s'arrête si un obstacle est à moins de `distance_min` mm.
    Retourne True si le trajet est complet, False si un obstacle l'a arrêté."""
    vitesse = max(0, min(255, int(vitesse)))
    bot.Ctrl_Ulatist_Switch(1)  # allume le capteur ultrason
    time.sleep(0.2)             # attend la première mesure
    try:
        move_forward(vitesse)
        # time.time() donne l'heure actuelle en secondes. On calcule l'heure de fin,
        # puis on boucle jusqu'à l'atteindre. Contrairement à time.sleep(duree),
        # cela permet de vérifier la distance PENDANT que le robot roule.
        fin = time.time() + duree
        while time.time() < fin:
            distance = lire_distance_mm()
            # distance > 0 : on ignore les lectures à 0, qui sont en général des erreurs de mesure
            if 0 < distance < distance_min:
                print(f"Obstacle à {distance} mm : arrêt")
                return False  # return quitte la fonction... mais le finally s'exécute quand même
            time.sleep(0.05)  # vérifie environ 20 fois par seconde
        return True
    finally:
        stop_robot()
        bot.Ctrl_Ulatist_Switch(0)


# Distance parcourue en 1 seconde à la vitesse VITESSE_SOL.
# Pour calibrer : exécute avancer(duree=1), mesure la distance avec une règle,
# et remplace 25 par ta mesure. À refaire si la batterie est faible, si le sol
# change ou si tu modifies VITESSE_SOL.
CM_PAR_SECONDE = 25


def avancer_cm(distance_cm, vitesse=VITESSE_SOL):
    """Avance d'environ `distance_cm` centimètres (précis seulement si CM_PAR_SECONDE est calibré
    pour cette vitesse)."""
    # temps = distance / vitesse  (ex. 30 cm / 25 cm/s = 1,2 s)
    avancer(vitesse, distance_cm / CM_PAR_SECONDE)

### 1.5 Photos
La caméra USB est lue avec **OpenCV** (`cv2`). Les photos sont enregistrées en `.jpg` dans le dossier `photos/`, à côté du notebook.

In [5]:
import glob        # liste des fichiers correspondant à un motif (ex. /dev/video*)
import subprocess  # exécute des commandes Linux depuis Python


def ouvrir_camera():
    """Cherche une caméra qui fonctionne et retourne (camera, nom), ou (None, None) si aucune.

    Sous Linux, chaque caméra apparaît comme un fichier /dev/videoN. Sur le Raspberry Pi 5,
    il existe aussi des /dev/video19, /dev/video20... qui ne sont PAS des caméras
    (ce sont des circuits internes de traitement vidéo). La caméra USB n'est donc
    pas forcément /dev/video0 : on les essaie toutes, dans l'ordre."""
    # Trie par numéro (video2 avant video10) ; on ignore les noms sans numéro
    candidats = [p for p in glob.glob("/dev/video*") if p[len("/dev/video"):].isdigit()]
    candidats.sort(key=lambda p: int(p[len("/dev/video"):]))
    for chemin in candidats:
        # CAP_V4L2 : on force le pilote vidéo Linux standard (Video4Linux2)
        camera = cv2.VideoCapture(chemin, cv2.CAP_V4L2)
        if camera.isOpened():
            ok, _ = camera.read()  # le fichier s'ouvre parfois sans être une vraie caméra :
            if ok:                 # on vérifie qu'on peut réellement lire une image
                return camera, chemin
        camera.release()           # pas une caméra utilisable : on la referme
    return None, None


def diagnostic_camera():
    """Affiche des informations pour comprendre pourquoi la caméra ne s'ouvre pas."""
    # (titre, commande Linux) ; 2>&1 regroupe les messages d'erreur avec le reste
    commandes = [
        ("Fichiers vidéo présents", "ls -l /dev/video* 2>&1 | head -20"),
        ("Périphériques USB (la caméra doit apparaître)", "lsusb"),
        ("Caméras détectées", "v4l2-ctl --list-devices 2>&1"),
        ("Programmes qui utilisent une caméra",
         "fuser -v /dev/video* 2>&1 | grep -v '^$' || echo aucun"),
        ("Programmes Python en cours", "ps aux | grep -i python | grep -v grep"),
    ]
    for titre, commande in commandes:
        print(f"\n----- {titre} -----")
        # shell=True : la commande est interprétée par bash (permet | et 2>&1)
        # capture_output + text : récupère la sortie sous forme de texte
        resultat = subprocess.run(commande, shell=True, capture_output=True, text=True)
        print(resultat.stdout.strip() or resultat.stderr.strip() or "(rien)")

    camera, chemin = ouvrir_camera()
    if camera is None:
        print("\n==> Aucune caméra utilisable trouvée")
        return False
    camera.release()
    print(f"\n==> Caméra utilisable : {chemin}")
    return True


def prendre_photo(nb_photos=1, intervalle=0.5, dossier="photos", afficher=True):
    """Prend nb_photos photos avec la caméra et retourne la liste des fichiers créés.

    intervalle : secondes d'attente entre deux photos
    dossier    : dossier où enregistrer les photos (créé s'il n'existe pas)
    afficher   : True pour afficher chaque photo sous la cellule
    """
    # exist_ok=True : pas d'erreur si le dossier existe déjà
    os.makedirs(dossier, exist_ok=True)

    # Cherche et ouvre une caméra qui fonctionne (voir ouvrir_camera)
    camera, chemin_camera = ouvrir_camera()
    if camera is None:
        # Arrive si la caméra est débranchée ou déjà utilisée par un autre programme/notebook
        print("Erreur : aucune caméra utilisable (débranchée ou déjà utilisée).")
        print("        Lance le test 'Diagnostic caméra' pour en savoir plus.")
        return []  # liste vide = aucune photo prise
    print(f"Caméra : {chemin_camera}")

    # Demande une résolution de 640 x 480 pixels
    camera.set(cv2.CAP_PROP_FRAME_WIDTH, 640)
    camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 480)

    fichiers = []  # on y ajoutera le chemin de chaque photo réussie
    try:
        # Au démarrage, la caméra ajuste son exposition : les premières images
        # sont souvent trop sombres. On en lit 5 sans les garder.
        for _ in range(5):
            camera.read()

        for i in range(nb_photos):
            # read() retourne deux valeurs :
            #   ok    : True si l'image a bien été lue
            #   image : l'image elle-même (un tableau de pixels, en couleurs BGR)
            ok, image = camera.read()
            if not ok:
                print(f"Erreur : photo {i + 1} non prise")
                continue  # passe directement à la photo suivante

            # Nom de fichier unique basé sur la date et l'heure, par ex. 20260917_143012_512.jpg
            # %f donne les microsecondes (6 chiffres) ; [:-3] enlève les 3 derniers
            # pour garder les millisecondes. Ainsi, deux photos rapprochées ont des noms différents.
            nom = datetime.now().strftime("%Y%m%d_%H%M%S_%f")[:-3] + ".jpg"
            chemin = os.path.join(dossier, nom)  # ex. photos/20260917_143012_512.jpg
            cv2.imwrite(chemin, image)           # enregistre l'image sur le disque
            fichiers.append(chemin)
            print(f"Photo enregistrée : {chemin}")

            if afficher:
                display(Image(filename=chemin, width=320))  # aperçu réduit sous la cellule
            if i < nb_photos - 1:  # pas d'attente inutile après la dernière photo
                time.sleep(intervalle)
    finally:
        # Libère TOUJOURS la caméra. Sinon, elle resterait bloquée et les
        # appels suivants (ou les autres notebooks) ne pourraient plus l'ouvrir.
        camera.release()

    return fichiers


def test_photos_servo():
    """Tourne la caméra et prend une photo à gauche, au centre et à droite.
    Retourne False si au moins une photo a échoué."""
    reussi = True
    for angle in (135, 90, 45):  # 135 = gauche, 90 = centre, 45 = droite
        bot.Ctrl_Servo(1, angle)
        time.sleep(0.6)          # attend que le servo soit immobile, sinon la photo est floue
        if not prendre_photo():  # une liste vide est considérée comme False
            reussi = False
    bot.Ctrl_Servo(1, 90)        # remet la caméra au centre
    return reussi

print("Fonctions chargées")

Fonctions chargées


## 2. Tests
Exécute seulement les cellules que tu veux tester.

### Buzzer

In [ ]:
test_buzzer()  # un bip court

### Lumières (7 couleurs)

In [ ]:
test_lumieres()  # rouge, vert, bleu, jaune, violet, cyan, blanc

### Servos de la caméra

In [ ]:
test_servos()  # gauche, droite, centre, puis bas, haut, centre

### Moteurs un par un — ⚠️ **soulève le robot** (roues dans le vide)

In [ ]:
test_moteurs()  # note quelle roue tourne pour chaque numéro affiché

### Déplacements — ⚠️ **robot au sol, 1 m d'espace libre**

In [ ]:
test_deplacements()  # avancer, reculer, glisser gauche/droite, tourner gauche/droite

### Capteur ultrason (mets ta main devant)

In [ ]:
test_ultrason()  # 10 mesures en 3 secondes

### Capteurs de ligne (passe-les sur du noir / blanc)

In [ ]:
test_capteurs_ligne()  # 10 lectures en 3 secondes

### Avancer 1 s — ⚠️ robot au sol

In [ ]:
avancer(vitesse=VITESSE_SOL, duree=1)  # essaie 80, 120, 150 si le robot ne bouge pas

### Avancer 3 s avec arrêt devant un obstacle — ⚠️ robot au sol

In [ ]:
# Affiche True si le trajet est complet, False si un obstacle à moins de 15 cm l'a arrêté
avancer_securise(vitesse=VITESSE_SOL, duree=3)

### Avancer d'une distance (calibre `CM_PAR_SECONDE` avant)

In [ ]:
avancer_cm(30)  # environ 30 cm

### Diagnostic caméra (si les photos ne marchent pas)

In [ ]:
diagnostic_camera()

### Prendre une photo

In [ ]:
prendre_photo()  # la photo s'affiche sous la cellule

### Photos gauche / centre / droite

In [ ]:
test_photos_servo()

## 3. Menu interactif (tous les tests)
Exécute cette cellule, puis tape le numéro d'un test dans la zone de saisie qui apparaît et appuie sur Entrée.
Tape `q` pour quitter. Le bouton ■ (*Interrupt*) arrête le menu et le robot.

In [6]:
# Dictionnaire des tests : clé = ce que l'utilisateur tape, valeur = (description, fonction à appeler).
# On met la fonction SANS parenthèses (test_buzzer et non test_buzzer()) :
# on la stocke pour l'appeler plus tard, au lieu de l'exécuter tout de suite.
# lambda: ... crée une petite fonction sans nom, utile quand on veut
# appeler une fonction avec des paramètres précis (ex. duree=3).
TESTS = {
    "1": ("Buzzer", test_buzzer),
    "2": ("Lumières", test_lumieres),
    "3": ("Servos caméra", test_servos),
    "4": ("Moteurs un par un (SOULEVER LE ROBOT)", test_moteurs),
    "5": ("Déplacements (robot au sol, espace libre)", test_deplacements),
    "6": ("Capteur ultrason", test_ultrason),
    "7": ("Capteurs de ligne", test_capteurs_ligne),
    "8": (f"Avancer 1 s à vitesse {VITESSE_SOL} (robot au sol)", avancer),
    "9": ("Avancer 3 s avec arrêt devant obstacle (robot au sol)", lambda: avancer_securise(duree=3)),
    "10": ("Avancer d'environ 30 cm (robot au sol, CM_PAR_SECONDE calibré)", lambda: avancer_cm(30)),
    "11": ("Prendre une photo", prendre_photo),
    "12": ("Prendre 3 photos (1 s d'écart)", lambda: prendre_photo(nb_photos=3, intervalle=1)),
    "13": ("Photos gauche / centre / droite", test_photos_servo),
    "14": ("Tout arrêter", tout_arreter),
    "15": ("Diagnostic caméra", diagnostic_camera),
    "v": ("Changer la vitesse au sol", lambda: changer_vitesse()),
}


def changer_vitesse():
    """Demande une nouvelle vitesse et l'applique aux tests de déplacement."""
    # global : on modifie la variable VITESSE_SOL définie en dehors de la fonction
    global VITESSE_SOL
    texte = input(f"Nouvelle vitesse (actuelle {VITESSE_SOL}, 0 à 255) : ").strip()
    if not texte.isdigit():  # isdigit() : vrai seulement si le texte ne contient que des chiffres
        print("  Vitesse invalide")
        return False
    VITESSE_SOL = min(255, int(texte))
    # Les valeurs par défaut des fonctions sont fixées au moment de leur définition :
    # on remplace donc les tests de déplacement pour qu'ils utilisent la nouvelle vitesse.
    TESTS["5"] = (f"Déplacements à vitesse {VITESSE_SOL} (robot au sol, espace libre)",
                  lambda: test_deplacements(vitesse=VITESSE_SOL))
    TESTS["8"] = (f"Avancer 1 s à vitesse {VITESSE_SOL} (robot au sol)",
                  lambda: avancer(vitesse=VITESSE_SOL))
    TESTS["9"] = (f"Avancer 3 s à vitesse {VITESSE_SOL} avec arrêt devant obstacle (robot au sol)",
                  lambda: avancer_securise(vitesse=VITESSE_SOL, duree=3))
    TESTS["10"] = (f"Avancer d'environ 30 cm à vitesse {VITESSE_SOL} (robot au sol)",
                   lambda: avancer_cm(30, vitesse=VITESSE_SOL))
    print(f"  Vitesse au sol : {VITESSE_SOL}")


def menu():
    """Affiche la liste des tests en boucle et exécute celui choisi, jusqu'à ce qu'on tape q."""
    try:
        while True:  # boucle infinie : on en sort avec break (choix q) ou une interruption
            print("\n=== Test RaspBot V2 ===")
            # .items() donne les paires (clé, valeur) ; la valeur est elle-même une paire
            # (nom, fonction), qu'on décompose. _ signifie qu'on n'utilise pas la fonction ici.
            for cle, (nom, _) in TESTS.items():
                print(f" {cle:>2}. {nom}")  # :>2 aligne les numéros à droite sur 2 caractères
            print("  q. Quitter")

            # input() attend que l'utilisateur tape quelque chose.
            # .strip() enlève les espaces autour, .lower() met en minuscules (Q devient q).
            choix = input("Choix : ").strip().lower()
            if choix == "q":
                break  # sort de la boucle while
            if choix not in TESTS:
                print("  Choix invalide")
                continue  # recommence la boucle sans rien exécuter

            nom, fonction = TESTS[choix]
            print(f"--> {nom}")
            try:
                resultat = fonction()  # les parenthèses exécutent enfin la fonction choisie
                # Les fonctions qui peuvent échouer retournent False ou une liste vide ([]).
                # Les autres ne retournent rien (None) : on considère alors que c'est réussi.
                if resultat is None or resultat:
                    print("  OK")
                else:
                    print("  ÉCHEC (voir le message ci-dessus)")
            except Exception as erreur:
                # Une erreur dans un test n'arrête pas le menu : on l'affiche et on continue.
                print(f"  Erreur : {erreur}")
                tout_arreter()
    except KeyboardInterrupt:
        # Levée quand on clique sur ■ (Interrupt) dans Jupyter, ou Ctrl+C dans un terminal
        print("\nInterrompu")
    finally:
        # Quelle que soit la façon de quitter le menu, le robot est remis dans un état sûr
        tout_arreter()
        print("Robot arrêté.")


menu()


=== Test RaspBot V2 ===
  1. Buzzer
  2. Lumières
  3. Servos caméra
  4. Moteurs un par un (SOULEVER LE ROBOT)
  5. Déplacements (robot au sol, espace libre)
  6. Capteur ultrason
  7. Capteurs de ligne
  8. Avancer 1 s à vitesse 100 (robot au sol)
  9. Avancer 3 s avec arrêt devant obstacle (robot au sol)
 10. Avancer d'environ 30 cm (robot au sol, CM_PAR_SECONDE calibré)
 11. Prendre une photo
 12. Prendre 3 photos (1 s d'écart)
 13. Photos gauche / centre / droite
 14. Tout arrêter
 15. Diagnostic caméra
  v. Changer la vitesse au sol
  q. Quitter


Choix :  1


--> Buzzer
  OK

=== Test RaspBot V2 ===
  1. Buzzer
  2. Lumières
  3. Servos caméra
  4. Moteurs un par un (SOULEVER LE ROBOT)
  5. Déplacements (robot au sol, espace libre)
  6. Capteur ultrason
  7. Capteurs de ligne
  8. Avancer 1 s à vitesse 100 (robot au sol)
  9. Avancer 3 s avec arrêt devant obstacle (robot au sol)
 10. Avancer d'environ 30 cm (robot au sol, CM_PAR_SECONDE calibré)
 11. Prendre une photo
 12. Prendre 3 photos (1 s d'écart)
 13. Photos gauche / centre / droite
 14. Tout arrêter
 15. Diagnostic caméra
  v. Changer la vitesse au sol
  q. Quitter


Choix :  8


--> Avancer 1 s à vitesse 100 (robot au sol)
  OK

=== Test RaspBot V2 ===
  1. Buzzer
  2. Lumières
  3. Servos caméra
  4. Moteurs un par un (SOULEVER LE ROBOT)
  5. Déplacements (robot au sol, espace libre)
  6. Capteur ultrason
  7. Capteurs de ligne
  8. Avancer 1 s à vitesse 100 (robot au sol)
  9. Avancer 3 s avec arrêt devant obstacle (robot au sol)
 10. Avancer d'environ 30 cm (robot au sol, CM_PAR_SECONDE calibré)
 11. Prendre une photo
 12. Prendre 3 photos (1 s d'écart)
 13. Photos gauche / centre / droite
 14. Tout arrêter
 15. Diagnostic caméra
  v. Changer la vitesse au sol
  q. Quitter


Choix :  9


--> Avancer 3 s avec arrêt devant obstacle (robot au sol)
  OK

=== Test RaspBot V2 ===
  1. Buzzer
  2. Lumières
  3. Servos caméra
  4. Moteurs un par un (SOULEVER LE ROBOT)
  5. Déplacements (robot au sol, espace libre)
  6. Capteur ultrason
  7. Capteurs de ligne
  8. Avancer 1 s à vitesse 100 (robot au sol)
  9. Avancer 3 s avec arrêt devant obstacle (robot au sol)
 10. Avancer d'environ 30 cm (robot au sol, CM_PAR_SECONDE calibré)
 11. Prendre une photo
 12. Prendre 3 photos (1 s d'écart)
 13. Photos gauche / centre / droite
 14. Tout arrêter
 15. Diagnostic caméra
  v. Changer la vitesse au sol
  q. Quitter

Interrompu
Robot arrêté.


## 🛑 Arrêt d'urgence
Exécute cette cellule si le robot fait n'importe quoi, et à la fin de ta séance.

In [3]:
tout_arreter()              # moteurs, buzzer et lumières éteints
bot.Ctrl_Ulatist_Switch(0)  # capteur ultrason éteint
bot.Ctrl_Servo(1, 90)       # caméra recentrée horizontalement
bot.Ctrl_Servo(2, 90)       # et verticalement
print("Robot arrêté")

Robot arrêté
